<img src="../images/nexrad-level3-houston-storm-total.png" width=600 alt="KHGX storm-total precipitation on 22 March 2022"></img>

# NEXRAD Level III Products from AWS
---

## Overview

The other NEXRAD notebooks in this cookbook read **Level II** data: the full radar volume, every sweep and every moment, about 18 MB per volume. The same radars also publish **Level III** products, generated at each site by the Radar Product Generator. A Level III file holds a single product (one tilt of one moment, or a derived quantity such as one-hour rainfall or hydrometeor class), it is 50 to 300 times smaller than the volume it came from, and it is the form in which the National Weather Service itself distributes and displays NEXRAD data. The Unidata-managed `unidata-nexrad-level3` bucket on AWS Open Data carries these products for every WSR-88D within minutes of each scan, with an archive reaching back to early 2022.

This notebook works through the morning of **22 March 2022**, when a squall line crossed southeast Texas and produced two brief tornadoes near Houston, using the KHGX (Houston/Galveston) radar. We will:

1. Learn the product catalog and how the bucket is organized
1. List a directory-less bucket efficiently with key prefixes
1. Read base reflectivity and velocity at the time of a tornado
1. Compare a Level III reflectivity product with the Level II sweep it came from
1. Build a rainfall time series at points from the one-hour accumulation product
1. Map the storm-total accumulation and compare it with rain gauges
1. Find the newest scan for near-real-time use, including the "no precipitation" case

## Prerequisites
| Concepts | Importance | Notes |
| --- | --- | --- |
| [Py-ART Basics](../foundations/pyart-basics) | Required | Radar objects, fields, `RadarMapDisplay` |
| [Intro to Cartopy](https://foundations.projectpythia.org/core/cartopy/cartopy) | Required | Projections and features |
| [Matplotlib Basics](https://foundations.projectpythia.org/core/matplotlib/matplotlib-basics) | Required | Basic plotting |
| [Looking at NEXRAD Data from Moore, Oklahoma](moore-oklahoma-tornado) | Helpful | The Level II workflow this notebook complements |
| [Pandas](https://foundations.projectpythia.org/core/pandas/pandas) | Helpful | Tables and the time series |

- **Time to learn**: 30 minutes
---

## Imports

In [ ]:
import time
import warnings
from datetime import datetime, timedelta, timezone

import cartopy.crs as ccrs
import fsspec
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyart
from metpy.plots import USCOUNTIES

warnings.filterwarnings("ignore")

## Level II and Level III

A WSR-88D completes a volume scan every four to ten minutes. The raw moments from every sweep of that volume (reflectivity, velocity, spectrum width and the three dual-polarization moments) make up **Level II**. The Radar Product Generator at the site then turns the volume into **Level III products**: a few dozen files, each holding one tilt of one moment or one derived quantity, already quality-controlled, quantized, and ready to display. Each product has a three-character mnemonic and a numeric product code. The products you will meet most often, with the code Py-ART uses to identify them:

| Mnemonic | Code | Product | Resolution and range |
| --- | --- | --- | --- |
| `N0B`, `N1B`, `N2B`, `N3B` | 153 | Base reflectivity, super-resolution, lowest four tilts | 0.25 km x 0.5 deg, 460 km |
| `N0G`, `N1G` | 154 | Base velocity, super-resolution | 0.25 km x 0.5 deg, 300 km |
| `N0Q` | 94 | Base reflectivity, legacy resolution | 1 km x 1 deg, 460 km |
| `N0U` | 99 | Base velocity, legacy resolution | 0.25 km x 1 deg, 300 km |
| `N0X`, `N0C`, `N0K`, `N0H` | 159, 161, 163, 165 | Differential reflectivity, correlation coefficient, specific differential phase, hydrometeor classification | 0.25 km x 1 deg, 300 km |
| `DPR` | 176 | Instantaneous precipitation rate (dual-polarization) | 0.25 km x 1 deg, 230 km |
| `DAA` | 170 | One-hour precipitation accumulation (dual-polarization) | 0.25 km x 1 deg, 230 km |
| `DTA` | 172 | Storm-total precipitation accumulation (dual-polarization) | 0.25 km x 1 deg, 230 km |
| `OHA`, `NTP` | 169, 80 | One-hour and storm-total accumulation from the legacy single-polarization algorithm | 2 km x 1 deg, 230 km |
| `HHC` | 177 | Hybrid hydrometeor classification | 0.25 km x 1 deg, 230 km |

Py-ART's `pyart.io.read_nexrad_level3` reads every product above into an ordinary `Radar` object with a single sweep and a single field. Not every product in the bucket is covered: a few are graphics or tables rather than radial images (`NCR` composite reflectivity, `NST` storm tracks, `NMD` mesocyclone detections, `NET` echo tops, `NVL` VIL) and Py-ART raises `NotImplementedError` on them, while the enhanced echo tops (`EET`) and digital VIL (`DVL`) open but come back without a field. [MetPy's `Level3File`](https://unidata.github.io/MetPy/latest/api/generated/metpy.io.Level3File.html) decodes all of those.

Level III costs you the vertical structure (one tilt per file) and the ability to re-process the raw moments. In exchange you get files that read in a fraction of a second, derived products that would take real effort to reproduce from Level II (the dual-polarization rainfall estimates in particular), and the exact fields that forecasters look at.

## The `unidata-nexrad-level3` bucket

Every object in the bucket sits at the top level. There are no year or site folders; the whole structure is in the key:

```
{SITE}_{PRODUCT}_{YYYY}_{MM}_{DD}_{HH}_{mm}_{SS}
```

- `SITE` is the radar's four-letter identifier with the leading `K` (or `T` for a Terminal Doppler Weather Radar) removed, so KHGX becomes `HGX`.
- `PRODUCT` is the three-character mnemonic from the table above.
- The timestamp is the UTC start time of the volume scan the product was generated from, which is also the timestamp in the matching Level II file name.

A flat bucket with hundreds of products per radar every few minutes holds billions of keys, so listing it wholesale (`fs.ls(bucket)`) never finishes. The bucket is designed to be listed by **prefix**: S3 only walks the keys that match, so `HGX_DAA_2022_03_22_12` returns the ten one-hour accumulation files KHGX produced between 12:00 and 12:59 UTC and nothing else. `fsspec` exposes exactly this through the `prefix` argument of `find`.

We will use the same anonymous S3 filesystem as the Level II notebooks and wrap the listing in a small helper that returns a tidy table.

In [ ]:
BUCKET = "unidata-nexrad-level3"
fs = fsspec.filesystem("s3", anon=True)


def key_time(key):
    """UTC datetime encoded in a Level III key, SITE_PROD_YYYY_MM_DD_HH_MM_SS."""
    parts = key.split("_")
    return datetime(*map(int, parts[2:8]), tzinfo=timezone.utc)


def list_level3(site, product, prefix):
    """Level III files for one site and product whose timestamp starts with ``prefix``.

    ``prefix`` can be as coarse as a day ("2022_03_22") or as fine as a minute
    ("2022_03_22_12_20").  One bounded S3 LIST request either way.
    """
    found = fs.find(BUCKET, prefix=f"{site}_{product}_{prefix}", detail=True)
    rows = []
    for path, info in sorted(found.items()):
        key = path.split("/", 1)[1]
        rows.append({"key": key, "time": key_time(key), "bytes": info["size"]})
    return pd.DataFrame(rows)


list_level3("HGX", "DAA", "2022_03_22_12")

Reading is just as direct. `pyart.io.read_nexrad_level3` accepts a file-like object, so we open the key with `fsspec` and hand it over. Nothing is written to disk.

In [ ]:
def read_level3(key):
    """Read one Level III product straight from S3 into a Py-ART Radar object."""
    with fs.open(f"{BUCKET}/{key}", "rb") as f:
        return pyart.io.read_nexrad_level3(f)


radar = read_level3("HGX_N0B_2022_03_22_12_01_25")
print(f"fields:        {list(radar.fields)}")
print(f"sweeps:        {radar.nsweeps} at {radar.fixed_angle['data'][0]:.1f} degrees")
print(f"rays x gates:  {radar.nrays} x {radar.ngates}")
print(f"max range:     {radar.range['data'].max() / 1000:.0f} km")
print(f"scan time:     {radar.time['units']}")

## The storm: 22 March 2022

A warm front lifted north across Texas on 21 March 2022. Long-track supercells formed along the dryline that afternoon and produced tornadoes from south-central Texas into the northern counties of the Houston forecast area through the evening. Overnight a squall line organized along the cold front and crossed southeast Texas during the early morning of 22 March. The bowing segment of that line produced two brief tornadoes: an EF1 near Beasley in Fort Bend County at 5:35 AM CDT (10:35 UTC) that injured four people in an RV, and an EF0 at Danbury in Brazoria County at 7:21 AM CDT (12:21 UTC). Rain gauges across the Houston metro recorded one to two and a half inches in the few hours it took the line to pass. The National Weather Service's [event summary](https://www.weather.gov/hgx/20220321-tornadooutbreak) has the full tornado list and the gauge reports we will use later.

The [echo top height notebook](echo_top_height) in this cookbook reads the KHGX Level II volume from 12:01:25 UTC on this morning. We start from the same scan.

## Base reflectivity and velocity at the time of the Danbury tornado

The super-resolution reflectivity (`N0B`) and velocity (`N0G`) products from the 12:20:27 UTC volume are the last scan before the Danbury tornado touched down at 12:21 UTC. We read both and plot them side by side with the same map setup as the Moore, Oklahoma notebook, zoomed to Brazoria County. Py-ART names the fields `reflectivity` and `velocity`, and the `NWSRef` and `NWSVel` color tables (registered by the `cmweather` package) are the ones forecasters use.

In [ ]:
reflectivity = read_level3("HGX_N0B_2022_03_22_12_20_27")
velocity = read_level3("HGX_N0G_2022_03_22_12_20_27")

DANBURY = (29.229, -95.346)  # EF0 tornado, 12:21 UTC

fig = plt.figure(figsize=(14, 6))
panels = [
    (reflectivity, "reflectivity", "NWSRef", -10, 65, "Reflectivity (dBZ)"),
    (velocity, "velocity", "NWSVel", -40, 40, "Radial velocity (m/s)"),
]
for i, (rad, field, cmap, vmin, vmax, label) in enumerate(panels, start=1):
    ax = fig.add_subplot(1, 2, i, projection=ccrs.PlateCarree())
    display = pyart.graph.RadarMapDisplay(rad)
    display.plot_ppi_map(
        field,
        sweep=0,
        ax=ax,
        vmin=vmin,
        vmax=vmax,
        cmap=cmap,
        colorbar_label=label,
        title=f"KHGX {field} 0.5 deg, 2022-03-22 12:20:27 UTC",
        min_lon=-96.0,
        max_lon=-94.6,
        min_lat=28.8,
        max_lat=30.0,
        lat_lines=np.arange(28.75, 30.25, 0.25),
        lon_lines=np.arange(-96.0, -94.5, 0.25),
    )
    ax.add_feature(USCOUNTIES, linewidth=0.4)
    ax.plot(DANBURY[1], DANBURY[0], marker="v", color="black", markersize=9,
            transform=ccrs.PlateCarree())
    ax.annotate("Danbury EF0", DANBURY[::-1], xytext=(8, -4), textcoords="offset points")
plt.tight_layout()

The leading edge of the bow is over Brazoria County, with inbound low-level flow (green) behind it and outbound flow (red) ahead of it to the northeast. The Danbury tornado was 30 yards wide and lasted about a minute, far below what a 0.5 degree beam resolves at 35 km, so do not expect a textbook couplet at the marker: Level III velocity shows the mesoscale circulation that spawned it, not the tornado itself. That is a limitation of the radar geometry, not of Level III; the Level II velocity for this sweep is the same data.

## The same sweep, two files

That last sentence can be checked. The Level II volume that starts at 12:01:25 UTC and the `N0B` product with the same timestamp come from the same antenna rotation. We read both, time the reads, and compare the lowest sweep.

In [ ]:
LEVEL2_KEY = "unidata-nexrad-level2/2022/03/22/KHGX/KHGX20220322_120125_V06"
LEVEL3_KEY = "HGX_N0B_2022_03_22_12_01_25"

t0 = time.perf_counter()
level2 = pyart.io.read_nexrad_archive(f"s3://{LEVEL2_KEY}")
level2_seconds = time.perf_counter() - t0

t0 = time.perf_counter()
level3 = read_level3(LEVEL3_KEY)
level3_seconds = time.perf_counter() - t0

sweep0 = level2.extract_sweeps([0])
z2 = sweep0.fields["reflectivity"]["data"]
z3 = level3.fields["reflectivity"]["data"]

pd.DataFrame(
    {
        "Level II volume": [
            f"{fs.info(LEVEL2_KEY)['size'] / 1e6:.1f} MB",
            f"{level2_seconds:.1f} s",
            level2.nsweeps,
            len(level2.fields),
            f"{z2.shape[0]} x {z2.shape[1]}",
            int(z2.count()),
            f"{z2.max():.1f} dBZ",
        ],
        "Level III N0B": [
            f"{fs.info(f'{BUCKET}/{LEVEL3_KEY}')['size'] / 1e6:.2f} MB",
            f"{level3_seconds:.1f} s",
            level3.nsweeps,
            len(level3.fields),
            f"{z3.shape[0]} x {z3.shape[1]}",
            int(z3.count()),
            f"{z3.max():.1f} dBZ",
        ],
    },
    index=["file size", "read time", "sweeps", "fields", "lowest sweep rays x gates",
           "valid gates in lowest sweep", "maximum reflectivity"],
)

The lowest sweep has the same number of valid gates and the same maximum in both files. The Level III product is the Level II sweep after the Radar Product Generator's processing: the same 0.5 dB quantization, the same 720 super-resolution radials, eight more range gates because the product's range axis starts at the antenna rather than at the first Level II gate. Everything else in the volume (the other 20 sweeps, velocity, spectrum width and the dual-polarization moments) is what the 50-fold difference in size and read time buys.

A side-by-side plot of the two makes the point visually.

In [ ]:
fig = plt.figure(figsize=(14, 6))
for i, (rad, title) in enumerate(
    [(sweep0, "Level II, sweep 0"), (level3, "Level III, N0B")], start=1
):
    ax = fig.add_subplot(1, 2, i, projection=ccrs.PlateCarree())
    display = pyart.graph.RadarMapDisplay(rad)
    display.plot_ppi_map(
        "reflectivity",
        sweep=0,
        ax=ax,
        vmin=-10,
        vmax=65,
        cmap="NWSRef",
        colorbar_label="Reflectivity (dBZ)",
        title=f"KHGX {title}, 2022-03-22 12:01:25 UTC",
        min_lon=-96.4,
        max_lon=-94.4,
        min_lat=28.8,
        max_lat=30.6,
        lat_lines=np.arange(28.5, 31, 0.5),
        lon_lines=np.arange(-96.5, -94, 0.5),
    )
    ax.add_feature(USCOUNTIES, linewidth=0.4)
plt.tight_layout()

## Rainfall at a point through the morning

The dual-polarization **one-hour accumulation** (`DAA`) is where Level III earns its keep. The product is the output of the WSR-88D's quantitative precipitation estimation chain: hydrometeor classification removes non-meteorological echoes, a rate is estimated gate by gate from reflectivity, differential reflectivity and specific differential phase, and the rate is integrated over the trailing hour. Reproducing that from Level II is a project; reading it is one line. Py-ART stores the field as `radar_estimated_rain_rate` with units of inches per hour, which over a one-hour window is the same number as inches of accumulation.

Py-ART computes a latitude and longitude for every gate, so "the accumulation at this address" is the value of the nearest gate. The helper below returns zero rather than a masked value where the product reports no precipitation, which is what the question "how much rain fell here?" needs.

In [ ]:
def distance_km(radar, lat, lon):
    """Great-circle distance in km from every gate of ``radar`` to a point."""
    gate_lat = np.radians(radar.gate_latitude["data"])
    gate_lon = np.radians(radar.gate_longitude["data"])
    lat, lon = np.radians(lat), np.radians(lon)
    h = (np.sin((gate_lat - lat) / 2) ** 2
         + np.cos(lat) * np.cos(gate_lat) * np.sin((gate_lon - lon) / 2) ** 2)
    return 2 * 6371.0 * np.arcsin(np.sqrt(h))


def value_at(radar, lat, lon, field="radar_estimated_rain_rate"):
    """Field value in the gate nearest to (lat, lon); 0.0 where the product is masked."""
    data = radar.fields[field]["data"]
    nearest = np.unravel_index(np.argmin(distance_km(radar, lat, lon)), data.shape)
    value = data[nearest]
    return 0.0 if np.ma.is_masked(value) else float(value)


PLACES = {
    "Conroe": (30.312, -95.456),
    "Bush Intercontinental": (29.984, -95.341),
    "Downtown Houston": (29.760, -95.370),
    "Hobby Airport": (29.645, -95.279),
    "Galveston": (29.301, -94.797),
}

The squall line reached the northern suburbs around 10:30 UTC and cleared the coast after 14:00 UTC. Listing the `DAA` files hour by hour over that window and reading each one gives about sixty files, each a quarter of a second to read.

In [ ]:
records = []
for hour in range(9, 16):
    for key in list_level3("HGX", "DAA", f"2022_03_22_{hour:02d}")["key"]:
        radar = read_level3(key)
        row = {name: value_at(radar, *latlon) for name, latlon in PLACES.items()}
        row["time"] = key_time(key)
        records.append(row)

one_hour = pd.DataFrame(records).set_index("time")
print(f"{len(one_hour)} one-hour accumulation products")
one_hour.round(2).iloc[::6]

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
one_hour.plot(ax=ax, linewidth=2)
ax.set_ylabel("One-hour accumulation (inches)")
ax.set_xlabel("Time (UTC), 22 March 2022")
ax.set_title("KHGX Level III DAA: trailing one-hour rainfall at five points")
ax.axvline(datetime(2022, 3, 22, 10, 35, tzinfo=timezone.utc), color="gray", linestyle=":")
ax.axvline(datetime(2022, 3, 22, 12, 21, tzinfo=timezone.utc), color="gray", linestyle=":")
ax.set_ylim(0, 2.0)
ax.text(datetime(2022, 3, 22, 10, 37, tzinfo=timezone.utc), 1.85, "Beasley EF1", color="gray")
ax.text(datetime(2022, 3, 22, 12, 23, tzinfo=timezone.utc), 1.85, "Danbury EF0", color="gray")
ax.grid(alpha=0.3)
ax.legend(loc="upper left");

The curves are the line itself, read from the ground. Conroe, 60 km north of downtown, peaks first at about 11:30 UTC; Bush Intercontinental follows half an hour later, downtown by 12:30, Hobby by 13:00, and Galveston on the coast at 14:00. Each curve rises as rain falls and decays over the following hour as the trailing window slides past it. The two tornado times sit exactly where the line passes the corresponding counties.

Hobby airport and downtown Houston are 15 km apart and their peak one-hour totals differ by a quarter, which is the kind of gradient a single rain gauge cannot see and a 1 km radar product can.

## Storm-total accumulation and the rain gauges

The **storm-total accumulation** (`DTA`) integrates the same rate field from the start of the event until an hour has passed with no rain. The last `DTA` before the KHGX umbrella dried out on this day is the 17:59:48 UTC product, so it holds the whole squall line. We map it over the metro area and mark the five airports whose Automated Surface Observing Systems reported daily rainfall in the NWS event summary.

In [ ]:
storm_total = read_level3("HGX_DTA_2022_03_22_17_59_48")

# ASOS daily precipitation through 6:53 PM CDT on 22 March 2022, from the NWS event summary.
GAUGES = pd.DataFrame(
    {
        "lat": [29.984, 29.645, 30.062, 29.622, 30.352],
        "lon": [-95.341, -95.279, -95.553, -95.657, -95.415],
        "gauge_inches": [1.51, 1.26, 1.33, 1.45, 1.42],
    },
    index=["Bush Intercontinental (KIAH)", "Hobby (KHOU)", "Hooks (KDWH)",
           "Sugar Land (KSGR)", "Conroe (KCXO)"],
)

fig = plt.figure(figsize=(9, 8))
ax = fig.add_subplot(1, 1, 1, projection=ccrs.PlateCarree())
display = pyart.graph.RadarMapDisplay(storm_total)
display.plot_ppi_map(
    "radar_estimated_rain_rate",
    sweep=0,
    ax=ax,
    vmin=0,
    vmax=4,
    cmap="RRate11",
    colorbar_label="Storm-total accumulation (inches)",
    title="KHGX Level III DTA, 2022-03-22 17:59:48 UTC",
    min_lon=-96.2,
    max_lon=-94.4,
    min_lat=28.9,
    max_lat=30.6,
    lat_lines=np.arange(29, 31, 0.5),
    lon_lines=np.arange(-96.5, -94, 0.5),
)
ax.add_feature(USCOUNTIES, linewidth=0.4)
for name, g in GAUGES.iterrows():
    ax.plot(g.lon, g.lat, marker="o", color="black", markersize=6, transform=ccrs.PlateCarree())
    ax.annotate(f"{name.split(' (')[0]}\n{g.gauge_inches:.2f} in gauge", (g.lon, g.lat),
                xytext=(7, -4), textcoords="offset points", fontsize=9)

In [ ]:
GAUGES["radar_inches"] = [
    value_at(storm_total, g.lat, g.lon) for _, g in GAUGES.iterrows()
]
GAUGES["radar / gauge"] = GAUGES["radar_inches"] / GAUGES["gauge_inches"]
GAUGES.round(2)

The comparison is fair in time: the one-hour product stayed empty at all five airports from 15:00 UTC to midnight, so both the storm total and the daily gauge reading cover the same rain. It is not flattering everywhere. The radar matches the gauges at Hobby and Sugar Land but reports half again as much at Bush Intercontinental, Hooks and Conroe, where the line was most intense. Overestimation in the cores of convective lines is a known behavior of radar rainfall estimation (hail and large drops raise reflectivity faster than the rain rate, and the relationships behind the algorithm are tuned for stratiform rain), and it is why operational products such as MRMS and the NWS stage-IV analysis bias-correct radar against gauges before anyone uses the numbers for hydrology. A Level III accumulation is an excellent map of *where* and *when*; treat the magnitudes as a first estimate.

## Near-real-time: the newest scan

New products land in the bucket within minutes of the scan, which makes the Level III bucket a live feed as well as an archive. Finding the newest file is a prefix problem again. Listing `HGX_N0B_` returns the first thousand keys in lexical order, which are the oldest keys from 2022; the right move is to list today's date prefix, and yesterday's if today is empty.

Two things change when you read live data:

- Rain products exist only when there is rain. When the radar detects no precipitation anywhere in its coverage, the `DAA` and `DTA` files are a few hundred bytes long and carry a text message instead of a radial image. Py-ART raises `AssertionError` on them. That empty file **is** the answer, so the helper below returns `None` rather than walking back in time to find a product that decodes.
- The reflectivity products always decode, whatever the weather, so `N0B` is the right product for a "what does the radar see right now" picture.

In [ ]:
def latest_level3(site, product, days_back=3):
    """Newest file for a site and product, searching back one day prefix at a time."""
    now = datetime.now(timezone.utc)
    for back in range(days_back):
        day = (now - timedelta(days=back)).strftime("%Y_%m_%d")
        files = list_level3(site, product, day)
        if len(files):
            return files.iloc[-1]
    return None


def read_level3_or_none(key):
    """Read a Level III product, or return None for the 'no precipitation' message files."""
    try:
        return read_level3(key)
    except AssertionError:
        return None


SITE = "HGX"

newest_z = latest_level3(SITE, "N0B")
newest_rain = latest_level3(SITE, "DAA")
now = datetime.now(timezone.utc)
for label, newest in [("reflectivity N0B", newest_z), ("one-hour accumulation DAA", newest_rain)]:
    if newest is None:
        print(f"{label}: nothing in the last three days (check the site key)")
    else:
        age = (now - newest.time).total_seconds() / 60
        print(f"{label}: {newest.key}  {newest.bytes:,} bytes  {age:.0f} minutes old")

In [ ]:
if newest_rain is not None:
    rain = read_level3_or_none(newest_rain.key)
    if rain is None:
        print(f"K{SITE} detected no precipitation in the hour ending {newest_rain.time:%Y-%m-%d %H:%M} UTC.")
    else:
        acc = rain.fields["radar_estimated_rain_rate"]["data"]
        print(f"Hour ending {newest_rain.time:%Y-%m-%d %H:%M} UTC: "
              f"{int(acc.count()):,} gates with rain, maximum {acc.max():.2f} inches.")

if newest_z is not None:
    live = read_level3(newest_z.key)
    fig = plt.figure(figsize=(8, 7))
    ax = fig.add_subplot(1, 1, 1, projection=ccrs.PlateCarree())
    display = pyart.graph.RadarMapDisplay(live)
    display.plot_ppi_map(
        "reflectivity",
        sweep=0,
        ax=ax,
        vmin=-10,
        vmax=65,
        cmap="NWSRef",
        colorbar_label="Reflectivity (dBZ)",
        title=f"K{SITE} N0B, newest scan {newest_z.time:%Y-%m-%d %H:%M:%S} UTC",
        resolution="50m",
    )
    ax.add_feature(USCOUNTIES, linewidth=0.3)

Change `SITE` to the three-character key of any WSR-88D (`TLX` for Oklahoma City, `FTG` for Denver, `LOT` for Chicago) and the two cells above become a live view of that radar. The site table shipped with Py-ART, `pyart.io.nexrad_common.NEXRAD_LOCATIONS`, lists every site with its coordinates.

---
## Summary

Level III products are the NEXRAD volume after the site's own processing: one product per file, small, quick to read with `pyart.io.read_nexrad_level3`, and including derived fields such as dual-polarization rainfall that are hard to reproduce from Level II. The `unidata-nexrad-level3` bucket is flat, so list it by key prefix (site, product, and as much of the timestamp as you need) and never by directory. Using the 22 March 2022 Houston squall line we read reflectivity and velocity at the time of a tornado, confirmed that a Level III reflectivity product is the Level II lowest sweep gate for gate, built a rainfall time series at five points from the one-hour accumulation product, mapped the storm total and compared it with airport gauges, and found the newest scan in a way that handles the "no precipitation" files correctly.

### What's next?

The [Moore, Oklahoma](moore-oklahoma-tornado) and [echo top height](echo_top_height) notebooks show what the full Level II volume adds: the vertical structure and the raw moments. For rainfall over areas larger than one radar, the Multi-Radar Multi-Sensor (MRMS) system merges every radar in the network with gauges and models; it is also freely available on AWS.

## Resources and references

- [NEXRAD on AWS](https://registry.opendata.aws/noaa-nexrad/) on the Registry of Open Data, which lists the `unidata-nexrad-level3` and `unidata-nexrad-level2` buckets and their notification topics
- [NEXRAD Level III product descriptions](https://www.ncei.noaa.gov/products/radar/next-generation-weather-radar) at NOAA's National Centers for Environmental Information
- [`pyart.io.read_nexrad_level3`](https://arm-doe.github.io/pyart/API/generated/pyart.io.read_nexrad_level3.html) in the Py-ART documentation
- [cmweather](https://cmweather.readthedocs.io/), the source of the `NWSRef`, `NWSVel` and `RRate11` color tables
- [March 21-22, 2022 tornado outbreak](https://www.weather.gov/hgx/20220321-tornadooutbreak), NWS Houston/Galveston, for the tornado surveys and the rainfall reports used above
- Helmus, J. J. and Collis, S. M. (2016). The Python ARM Radar Toolkit (Py-ART), a library for working with weather radar data in the Python programming language. *Journal of Open Research Software*, 4(1), e25. https://doi.org/10.5334/jors.119